# Final experiment
Load development data, verify model definitions

In [2]:
from pathlib import Path
import sys
import importlib

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.modeling as modeling
importlib.reload(modeling)

DATA_FILE = PROJECT_ROOT / "data/processed/flights_weather_2024_top10.csv.gz"

data = pd.read_csv(DATA_FILE, dtype={"CRSDepTime": "string"}, parse_dates=["FlightDate"])
data["departure_hour"] = data["CRSDepTime"].astype(int) // 100

assert data["FlightDate"].dt.year.eq(2024).all()
assert data["Origin"].nunique() == 10
assert data["DepDel15"].isin([0, 1]).all()

data_hash = modeling.file_hash(DATA_FILE)
print(f"Loaded {len(data):,} development flights")

Loaded 2,335,170 development flights


In [6]:
checks = []

for family, candidates in modeling.CANDIDATES.items():
    for with_weather in [False, True]:
        for candidate_id, settings in enumerate(candidates):
            pipeline, columns = modeling.make_model(family, with_weather)
            pipeline.set_params(**settings)

            assert set(columns).issubset(data.columns)
            assert not {"Origin", "DepDel15", "DepDelay"}.intersection(columns)

            checks.append({
                "family": family,
                "weather": with_weather,
                "candidate": candidate_id,
                "settings": settings,
            })

display(pd.DataFrame(checks))
print("All 18 configurations constructed succesfully")

,family,weather,candidate,settings
0,logistic,False,0,{'logisticregression__C': 0.1}
1,logistic,False,1,{'logisticregression__C': 1.0}
2,logistic,False,2,{'logisticregression__C': 10.0}
3,logistic,True,0,{'logisticregression__C': 0.1}
4,logistic,True,1,{'logisticregression__C': 1.0}
5,logistic,True,2,{'logisticregression__C': 10.0}
6,forest,False,0,"{'randomforestclassifier__max_depth': 10, 'ran..."
7,forest,False,1,"{'randomforestclassifier__max_depth': 16, 'ran..."
8,forest,False,2,"{'randomforestclassifier__max_depth': 24, 'ran..."
9,forest,True,0,"{'randomforestclassifier__max_depth': 10, 'ran..."


All 18 configurations constructed succesfully


## Verify shared evaluation against original preliminary results

In [9]:
pipeline, columns = modeling.make_model("logistic", with_weather=False)
pipeline.set_params(logisticregression__C=1.0)

reproduced = modeling.evaluate_airports(data, pipeline, columns, test_airport="ATL")
original = pd.read_csv(PROJECT_ROOT / "outputs/baseline_pilot/logistic_regression_airport_val.csv")
original = original.loc[original["model"].eq("Flight only")]

assert len(original) == 9
assert original["airport"].is_unique

expected = original.set_index("airport")[modeling.METRICS].sort_index()
actual = reproduced.set_index("validation_airport")[modeling.METRICS].sort_index()
actual.index.name = expected.index.name

display((actual - expected).rename(columns = lambda name: f"{name}_difference"))

pd.testing.assert_frame_equal(
    actual,
    expected,
    check_exact=False,
    atol=1e-8,
    rtol=1e-6,
)

print("Passed: shared evaluation reproduces the original score")

ATL held out - CLT validated
ATL held out - DEN validated
ATL held out - DFW validated
ATL held out - LAS validated
ATL held out - LAX validated
ATL held out - LGA validated
ATL held out - ORD validated
ATL held out - PHX validated
ATL held out - SEA validated


,average_precision_difference,roc_auc_difference,log_loss_difference
airport,,,
CLT,0.000000e+00,0.0,0.000000e+00
DEN,0.000000e+00,0.0,0.000000e+00
DFW,0.000000e+00,0.0,0.000000e+00
LAS,0.000000e+00,0.0,5.551115e-17
LAX,5.551115e-17,0.0,0.000000e+00
LGA,0.000000e+00,0.0,0.000000e+00
ORD,5.551115e-17,0.0,0.000000e+00
PHX,0.000000e+00,0.0,5.551115e-17
SEA,2.775558e-17,0.0,0.000000e+00


Passed: shared evaluation reproduces the original score


## Tune models for each held out airport

In [11]:
importlib.reload(modeling)

TUNING_DIR = PROJECT_ROOT / "outputs/final_evaluation/tuning"
airports = sorted(data["Origin"].unique())

for airport in airports:
    print(f"=== Held out airport : {airport} ===", flush=True)
    scores = modeling.tune_airport(
        data,
        test_airport=airport,
        output_dir=TUNING_DIR,
        data_hash=data_hash,
    )

    assert len(scores) == 162
    scores.to_csv(TUNING_DIR / airport / "all_scores.csv", index=False)

print("Tuning completed for all airports")

=== Held out airport : ATL ===
Running: logistic_flight_only_0_cad961d41f552b8c
ATL held out - CLT validated
ATL held out - DEN validated
ATL held out - DFW validated
ATL held out - LAS validated
ATL held out - LAX validated
ATL held out - LGA validated
ATL held out - ORD validated
ATL held out - PHX validated
ATL held out - SEA validated
Running: logistic_flight_only_1_c394e3fa05c745ea
ATL held out - CLT validated
ATL held out - DEN validated
ATL held out - DFW validated
ATL held out - LAS validated
ATL held out - LAX validated
ATL held out - LGA validated
ATL held out - ORD validated
ATL held out - PHX validated
ATL held out - SEA validated
Running: logistic_flight_only_2_935ab7abd834a145
ATL held out - CLT validated
ATL held out - DEN validated
ATL held out - DFW validated
ATL held out - LAS validated
ATL held out - LAX validated
ATL held out - LGA validated
ATL held out - ORD validated
ATL held out - PHX validated
ATL held out - SEA validated
Running: logistic_weather_0_32359134c70

In [12]:
parts = []

for airport in airports:
    scores = pd.read_csv(TUNING_DIR / airport / "all_scores.csv")

    assert len(scores) == 162
    assert scores["test_airport"].eq(airport).all()

    for _, group in scores.groupby(["family", "variant", "candidate_id"]):
        assert len(group) == 9
        assert set(group["validation_airport"]) == set(airports) - {airport}

    parts.append(scores)

tuning_scores = pd.concat(parts, ignore_index=True)

summary = (
    tuning_scores
    .groupby(["test_airport", "family", "variant", "candidate_id", "settings"])
    [modeling.METRICS]
    .mean()
    .reset_index()
)

selected = (
    summary
    .sort_values(
        ["average_precision", "candidate_id"],
        ascending=[False, True],
    )
    .drop_duplicates(["test_airport", "family", "variant"])
    .sort_values(["test_airport", "family", "variant"])
    .reset_index(drop=True)
)

assert len(summary) == 180
assert len(selected) == 60

summary.to_csv(TUNING_DIR / "candidate_summary.csv", index=False)
selected.to_csv(TUNING_DIR / "selected_candidates.csv", index=False)

display(selected)

,test_airport,family,variant,candidate_id,settings,average_precision,roc_auc,log_loss
0,ATL,forest,flight_only,2,"{""randomforestclassifier__max_depth"": 24, ""ran...",0.242192,0.617423,0.459945
1,ATL,forest,weather,2,"{""randomforestclassifier__max_depth"": 24, ""ran...",0.252727,0.628357,0.453893
2,ATL,logistic,flight_only,0,"{""logisticregression__C"": 0.1}",0.242453,0.619545,0.468132
3,ATL,logistic,weather,0,"{""logisticregression__C"": 0.1}",0.258552,0.632127,0.467349
4,ATL,xgboost,flight_only,2,"{""xgbclassifier__max_depth"": 6}",0.247122,0.621681,0.459654
5,ATL,xgboost,weather,2,"{""xgbclassifier__max_depth"": 6}",0.260534,0.634447,0.452469
6,CLT,forest,flight_only,2,"{""randomforestclassifier__max_depth"": 24, ""ran...",0.240782,0.618856,0.452600
7,CLT,forest,weather,2,"{""randomforestclassifier__max_depth"": 24, ""ran...",0.249311,0.628932,0.447645
8,CLT,logistic,flight_only,0,"{""logisticregression__C"": 0.1}",0.239229,0.619262,0.459564
9,CLT,logistic,weather,0,"{""logisticregression__C"": 0.1}",0.256409,0.634367,0.458280


## Load 2025 test data and check structure

In [ ]:
TEST_FILE = PROJECT_ROOT / "data/processed/flights_weather_2025_top10.csv.gz"

test_data = pd.read_csv(TEST_FILE, dtype={"CRSDepTime": "string"}, parse_dates=["FlightDate"])
test_data["departure_hour"] = test_data["CRSDepTime"].astype(int) // 100

assert test_data["FlightDate"].dt.year.eq(2025).all()
assert set(test_data["Origin"]) == set(airports)
assert test_data["DepDel15"].isin([0, 1]).all()
assert data["FlightDate"].max() < test_data["FlightDate"].min()

for family in modeling.CANDIDATES:
    for with_weather in [False, True]:
        _, columns = modeling.make_model(family, with_weather)
        assert set(columns).issubset(test_data.columns)

test_data_hash = modeling.file_hash(TEST_FILE)

print(f"Loaded {len(test_data):,} test flights")
display(test_data.groupby("Origin").size().rename("flights").to_frame())

In [14]:
import json

importlib.reload(modeling)

selected = pd.read_csv(TUNING_DIR / "selected_candidates.csv")

assert len(selected) == 60
assert not selected.duplicated(["test_airport", "family", "variant"]).any()
assert set(selected["test_airport"]) == set(airports)

for _, selection in selected.iterrows():
    model, columns = modeling.make_model(
        selection["family"],
        with_weather=selection["variant"] == "weather",
    )
    model.set_params(**json.loads(selection["settings"]))

    assert set(columns).issubset(data.columns)
    assert set(columns).issubset(test_data.columns)

print("All 60 configs loaded")

All 60 configs loaded


## Test run - ATL, logres, no-weather

In [21]:
importlib.reload(modeling)

PREDICTION_DIR = PROJECT_ROOT / "outputs/final_evaluation/predictions"

first = selected.loc[
    selected["test_airport"].eq("ATL")
    & selected["family"].eq("logistic")
    & selected["variant"].eq("flight_only")
].iloc[0]

first_path = modeling.run_final_pair(data, test_data, first, PREDICTION_DIR, data_hash, test_data_hash)

print(f"Predictions saved at: {first_path}")

ATL: logistic unseen fitted
ATL: logistic reference fitted
Predictions saved at: /Users/alainbouscholte/Developer/Thesis/outputs/final_evaluation/predictions/4b1f387e877a33a4/ATL_logistic_flight_only.csv.gz


In [ ]:
prediction_files = []

for i, (_, selection) in enumerate(selected.iterrows(), start=1):
    print(f"{i}/{len(selected)}: {selection['test_airport']} {selection['family']} {selection['variant']}", flush=True)

    path = modeling.run_final_pair(data, test_data, selection, PREDICTION_DIR, data_hash, test_data_hash)

    prediction_files.append({
        "test_airport": selection["test_airport"],
        "family": selection["family"],
        "variant": selection["variant"],
        "prediction_file": str(path.relative_to(PROJECT_ROOT)),
    })

    pd.DataFrame(prediction_files).to_csv(PREDICTION_DIR / "manifest.csv", index=False)

print("all model pairs completed/saved")

## Final test results

In [23]:
from sklearn.metrics import average_precision_score, roc_auc_score, log_loss

manifest = pd.read_csv(PREDICTION_DIR / "manifest.csv")

assert len(manifest) == 60
assert not manifest.duplicated(["test_airport", "family", "variant"]).any()

rows = []

for entry in manifest.to_dict("records"):
    predictions = pd.read_csv(PROJECT_ROOT / entry["prediction_file"])
    actual = predictions["DepDel15"].astype(int)

    for condition in ["constant", "unseen", "reference"]:
        probability = predictions[condition]

        rows.append({
            "airport": entry["test_airport"],
            "family": entry["family"],
            "variant": entry["variant"],
            "condition": condition,
            "flights": len(actual),
            "delay_rate": actual.mean(),
            "average_precision": average_precision_score(actual, probability),
            "roc_auc": roc_auc_score(actual, probability),
            "log_loss": log_loss(actual, probability),
        })

results = pd.DataFrame(rows)
overall = results.groupby(["family", "variant", "condition"])[modeling.METRICS].mean().reset_index()

RESULTS_DIR = PROJECT_ROOT / "outputs/final_evaluation"
results.to_csv(RESULTS_DIR / "airport_results.csv", index=False)
overall.to_csv(RESULTS_DIR / "overall_results.csv", index=False)

display(overall.round(4))

,family,variant,condition,average_precision,roc_auc,log_loss
0,forest,flight_only,constant,0.2325,0.5000,0.5430
1,forest,flight_only,reference,0.3595,0.6554,0.5170
2,forest,flight_only,unseen,0.3547,0.6511,0.5190
3,forest,weather,constant,0.2325,0.5000,0.5430
4,forest,weather,reference,0.3706,0.6646,0.5138
5,forest,weather,unseen,0.3640,0.6582,0.5161
6,logistic,flight_only,constant,0.2325,0.5000,0.5430
7,logistic,flight_only,reference,0.3566,0.6540,0.5172
8,logistic,flight_only,unseen,0.3523,0.6503,0.5199
9,logistic,weather,constant,0.2325,0.5000,0.5430


## Performance by airport

In [25]:
airport_comparison = (
    results.pivot(
        index=["airport", "family", "variant"],
        columns="condition",
        values="average_precision",
    )
    .reset_index()
)

airport_comparison["reference_gain"] = airport_comparison["reference"] - airport_comparison["unseen"]
airport_comparison["gain_over_baseline"] = airport_comparison["unseen"] - airport_comparison["constant"]

airport_comparison.to_csv(RESULTS_DIR / "airport_ap_comparison.csv", index=False)

display(airport_comparison.loc[airport_comparison["variant"].eq("weather")].sort_values("reference_gain", ascending=False).round(4))

condition,airport,family,variant,constant,reference,unseen,reference_gain,gain_over_baseline
23,DFW,xgboost,weather,0.2858,0.4399,0.4170,0.0230,0.1311
19,DFW,forest,weather,0.2858,0.4308,0.4122,0.0186,0.1264
59,SEA,xgboost,weather,0.2117,0.3275,0.3136,0.0139,0.1020
41,LGA,xgboost,weather,0.2223,0.4007,0.3873,0.0134,0.1650
35,LAX,xgboost,weather,0.1812,0.2786,0.2661,0.0125,0.0849
49,PHX,forest,weather,0.2106,0.3294,0.3185,0.0110,0.1079
55,SEA,forest,weather,0.2117,0.3228,0.3120,0.0108,0.1003
17,DEN,xgboost,weather,0.2557,0.4528,0.4426,0.0103,0.1868
25,LAS,forest,weather,0.2321,0.3787,0.3686,0.0101,0.1366
29,LAS,xgboost,weather,0.2321,0.3859,0.3771,0.0088,0.1451


Check if weather helped improve model performance at airports

In [27]:
weather_comparison = (
    results.loc[results["condition"].eq("unseen")]
    .pivot(
        index=["airport", "family"],
        columns="variant",
        values="average_precision",
    )
    .reset_index()
)

weather_comparison["weather_gain"] = weather_comparison["weather"] - weather_comparison["flight_only"]

weather_comparison.to_csv(RESULTS_DIR / "airport_weather_gains.csv", index=False)

display(weather_comparison.sort_values(["weather_gain"]).round(4))

variant,airport,family,flight_only,weather,weather_gain
9,DFW,forest,0.4203,0.4122,-0.0080
17,LAX,xgboost,0.2714,0.2661,-0.0053
11,DFW,xgboost,0.4210,0.4170,-0.0040
15,LAX,forest,0.2678,0.2659,-0.0020
27,SEA,forest,0.3081,0.3120,0.0039
8,DEN,xgboost,0.4380,0.4426,0.0046
12,LAS,forest,0.3634,0.3686,0.0053
24,PHX,forest,0.3117,0.3185,0.0068
29,SEA,xgboost,0.3067,0.3136,0.0069
16,LAX,logistic,0.2663,0.2735,0.0072


## Uncertainty in paired performance differences

In [ ]:
import src.bootstrap as bootstrap
importlib.reload(bootstrap)

bootstrap_ap_difference = bootstrap.bootstrap_ap_difference
N_BOOT = 1000
BOOTSTRAP_SEED = 42


Run first comparison

In [31]:
entry = manifest.loc[
    manifest["test_airport"].eq("ATL")
    & manifest["family"].eq("xgboost")
    & manifest["variant"].eq("weather")
].iloc[0]

predictions = pd.read_csv(PROJECT_ROOT / entry["prediction_file"])

interval = bootstrap_ap_difference(predictions, first="reference", second="unseen")

display(pd.DataFrame([interval]).round(4))

,ap_difference,ci_lower,ci_upper,bootstrap_repetitions
0,0.005,0.0002,0.0094,1000


Extend to reference vs unseen performance for all six variants at all ten airports

In [ ]:
INTERVAL_DIR = RESULTS_DIR / "bootstrap_reference"
INTERVAL_DIR.mkdir(exist_ok=True)

interval_rows = []

for entry in manifest.to_dict("records"):
    name = f"{entry['test_airport']}_{entry['family']}_{entry['variant']}"
    checkpoint, signature, row = bootstrap.bootstrap_checkpoint(
        INTERVAL_DIR, name,
        {"predictions": PROJECT_ROOT / entry["prediction_file"]},
        first="reference", second="unseen", n_boot=N_BOOT, seed=BOOTSTRAP_SEED,
    )

    if row is None:
        predictions = pd.read_csv(PROJECT_ROOT / entry["prediction_file"])

        row = {
            "airport": entry["test_airport"],
            "family": entry["family"],
            "variant": entry["variant"],
            **bootstrap_ap_difference(predictions, first="reference", second="unseen", n_boot=N_BOOT, seed=BOOTSTRAP_SEED),
        }
        bootstrap.save_bootstrap_checkpoint(checkpoint, signature, row)

    interval_rows.append(row)
    print(f"Completed: {name}", flush=True)

reference_intervals = pd.DataFrame(interval_rows)
reference_intervals.to_csv(RESULTS_DIR / "reference_ap_intervals.csv", index=False)

display(reference_intervals.round(4))

Calculate equivalent intervals for weather vs no-weather

In [ ]:
WEATHER_INTERVAL_DIR = RESULTS_DIR / "bootstrap_weather"
WEATHER_INTERVAL_DIR.mkdir(exist_ok=True)

weather_rows = []

for (airport, family), group in manifest.groupby(["test_airport", "family"]):
    name = f"{airport}_{family}"
    prediction_files = {
        variant: PROJECT_ROOT / group.loc[group["variant"].eq(variant), "prediction_file"].item()
        for variant in ["flight_only", "weather"]
    }
    checkpoint, signature, row = bootstrap.bootstrap_checkpoint(
        WEATHER_INTERVAL_DIR, name, prediction_files,
        first="weather", second="flight_only", n_boot=N_BOOT, seed=BOOTSTRAP_SEED,
    )

    if row is None:
        paired = {}        

        for variant in ["flight_only", "weather"]:
            entry = group.loc[group["variant"].eq(variant)].iloc[0]
            paired[variant] = (
                pd.read_csv(PROJECT_ROOT / entry["prediction_file"], parse_dates=["FlightDate"])
                .set_index("test_row_id")
                .sort_index()
            )
            assert paired[variant].index.is_unique

        flight = paired["flight_only"]
        weather = paired["weather"]

        pd.testing.assert_frame_equal(
            flight[["FlightDate", "Origin", "DepDel15"]],
            weather[["FlightDate", "Origin", "DepDel15"]],
        )

        predictions = flight[["FlightDate", "DepDel15"]].copy()
        predictions["flight_only"] = flight["unseen"]
        predictions["weather"] = weather["unseen"]

        row = {
            "airport": airport,
            "family": family,
            **bootstrap_ap_difference(
                predictions, first="weather", second="flight_only",
                n_boot=N_BOOT, seed=BOOTSTRAP_SEED
            ),
        }
        bootstrap.save_bootstrap_checkpoint(checkpoint, signature, row)

    weather_rows.append(row)        
    print(f"Completed: {name}", flush=True)

weather_intervals = pd.DataFrame(weather_rows)
weather_intervals.to_csv(RESULTS_DIR / "weather_ap_intervals.csv", index=False)

display(weather_intervals.round(4))


## Spot check for old version of modeling.py used for tuning the models

In [4]:
from pathlib import Path
import sys, importlib
import pandas as pd
from sklearn.metrics import average_precision_score

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.modeling as modeling
importlib.reload(modeling)

TUNING_DIR = PROJECT_ROOT / "outputs/final_evaluation/tuning"
data = pd.read_csv(
    PROJECT_ROOT / "data/processed/flights_weather_2024_top10.csv.gz",
    dtype={"CRSDepTime": "string"},
    parse_dates=["FlightDate"],
)
data["departure_hour"] = data["CRSDepTime"].astype(int) // 100

def spot_check(test_airport, family, variant, candidate_id, val_airport):
    saved = pd.read_csv(TUNING_DIR / test_airport / "all_scores.csv")
    row = saved.query("family == @family and variant == @variant and "
                      "candidate_id == @candidate_id and validation_airport == @val_airport").iloc[0]
    est, cols = modeling.make_model(family, with_weather=(variant == "weather"))
    est.set_params(**modeling.CANDIDATES[family][candidate_id])
    dev = data[data["Origin"].ne(test_airport)]
    src = dev[dev["Origin"].ne(val_airport) & dev["FlightDate"].lt("2024-10-01")]
    tgt = dev[dev["Origin"].eq(val_airport) & dev["FlightDate"].ge("2024-10-01")]
    est.fit(src[cols], src["DepDel15"].astype(int))
    ap = average_precision_score(tgt["DepDel15"].astype(int), est.predict_proba(tgt[cols])[:, 1])
    print(f"{family:8s} {variant:11s} new={ap:.10f} saved={row['average_precision']:.10f} "
          f"diff={ap - row['average_precision']:.2e}", flush=True)

spot_check("ATL", "logistic", "flight_only", 0, "CLT")
spot_check("ATL", "forest",   "weather",     2, "CLT")
spot_check("ATL", "xgboost",  "weather",     2, "CLT")

logistic flight_only new=0.2567207061 saved=0.2567207061 diff=5.55e-17
forest   weather     new=0.2626308111 saved=0.2626308111 diff=0.00e+00
xgboost  weather     new=0.2664099057 saved=0.2664099057 diff=0.00e+00
